## This tends to solve Task3


In [89]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import json
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

In [42]:
# parameter settings: -- parameter loading from task1

products = ['A1', 'A2', 'A3', 'B1', 'B2']

# weekly demand 
demand_data = {
    'Weekly_Mean': [
        962,  
        1923, 
        2500, 
        1154, 
        1538  
    ],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18] 
}

demand_df = pd.DataFrame(demand_data, index=products) 

# the BOM of requirements of each Products (A, B) from each Parts (P1~P20)
bom_df = pd.read_csv('../Data/BOM.csv', index_col=0).T # for the later matrix multiplication dim align

# calculate the target capacity for products
Z_score = norm.ppf(0.995)

demand_df['Weekly_std'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
# round up to maintain integer
demand_df['Product_Capacity'] = np.ceil(demand_df['Weekly_Mean'] + demand_df['Weekly_std']*Z_score)


# Aggregate to parts demand
part_demand = pd.DataFrame()
part_demand['Part_Weekly_Capacity'] = bom_df.dot(demand_df['Product_Capacity'])

part_demand['Part_Weekly_Capacity'] = np.ceil(part_demand['Part_Weekly_Capacity']) 

# Fix: Strip whitespace from the index
part_demand.index = part_demand.index.str.strip()

# -------------------------------------------------------------------------------
# steps and time cost per step
path = "../Data"  # Go up one level then into Data
step = pd.read_csv(f"{path}/Steps.csv", index_col=0)
timecost = pd.read_csv(f"{path}/Timecost.csv", index_col=0)

productivity = 60 / timecost

# Efficiency
OEE = 0.9 * 0.98
A = 5 * 16 # this is the reason why this has a large differences with the other two
A_eff = A * OEE

# Machine Cost
machine = pd.read_csv("../Data/Equipment.csv", index_col='Equipment', usecols=['Equipment', 'Installed price', 'C1', 'C2', 'C3'])
K_m = machine.loc[:, 'Installed price']

In [39]:
f_demand = np.ceil(part_demand / 4)

In [44]:
from Solver import CapacityPlanningSolver

# transform data structure first
f_demand = f_demand['Part_Weekly_Capacity'].to_dict()
K_m_dict= K_m.to_dict()


# call solver class
solver = CapacityPlanningSolver()

# create eligibility matrix
all_steps = solver._extract_all_steps(step)
eligibility = {}
for machine in machine.index:
    eligibility[machine] = {}
    for s in all_steps:
        # Machine 'AB' can do steps 'A' and 'B'
        # Machine 'ABCD' can do steps 'A', 'B', 'C', 'D'
        eligibility[machine][s] = 1 if s in machine else 0

# run solver
results = solver.solve(
    part_demand=f_demand,
    step_df=step,
    productivity_df=productivity,
    K_m=K_m,
    A_eff=A_eff,
    eligibility=eligibility,
    verbose=True
)

CAPACITY PLANNING OPTIMIZATION

Building optimization model...
Model Statistics:
  - Variables: 6786
  - Constraints: 5994
  - Demand-step combinations: 88

Solving MIP model...
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/tianyihu/.venvs/general/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/1q/yvvtgq1134x569hhykb6ll100000gn/T/b84966df987642688127738c1ad951ac-pulp.mps -sec 300 -timeMode elapsed -branch -printingOptions all -solution /var/folders/1q/yvvtgq1134x569hhykb6ll100000gn/T/b84966df987642688127738c1ad951ac-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 5999 COLUMNS
At line 19006 RHS
At line 25001 BOUNDS
At line 25028 ENDATA
Problem MODEL has 5994 rows, 6786 columns and 12928 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 300
Option for timeMode changed from cpu to elapsed
Continuous objective value is 5.32021e+07 - 0.00 second

In [ ]:
for _, row in results['machine_allocation'].iterrows():
            print(f"{row['Machine']:<12} {row['Quantity']:<10} "
                    f"${row['Installation_Cost']:<14,.0f} {row['Utilization_%']:<11.1f}%")

print("-" * 80)
print(f"Total Machines: {results['machine_allocation']['Quantity'].sum()}")

A            4          $600,000        100.0      %
ABC          1          $300,000        87.6       %
B            8          $1,600,000      100.0      %
C            2          $500,000        100.0      %
CD           1          $350,000        100.0      %
D            18         $5,400,000      100.0      %
E            8          $3,200,000      100.0      %
EG           1          $450,000        88.1       %
F            11         $4,400,000      94.6       %
G            2          $800,000        100.0      %
H            13         $13,000,000     95.3       %
I            13         $6,500,000      93.6       %
J            22         $16,500,000     96.4       %
K            2          $160,000        100.0      %
KLM          1          $150,000        52.0       %
L            9          $900,000        100.0      %
M            13         $650,000        100.0      %
--------------------------------------------------------------------------------
Total Machines: 12

In [84]:
print(f"the item of X_vars:\n", list(results['x_vars'].items())[10])
print(f"\nthe item of Y_vars:\n", list(results['y_vars'].items())[0])
print(f"\n exact value for time machine A allocate to K step:\n", results['x_vars'][('A', 'P1', 'K')].varValue )

the item of X_vars:
 (('A', 'P1', 'K'), x_('A',_'P1',_'K'))

the item of Y_vars:
 ('A', y_A)

 exact value for time machine A allocate to K step:
 0.0


In [85]:
def get_machine_time_allocation_dict(results):

    """

    dict : Dictionary with machine types as keys, containing ONLY numeric values:
        - quantity: number of machines (int)
        - total_hours: total machine-hours allocated (float)
        - total_hours_per_workday: hours divided by (5*16) then multiplied by 60 (float)
        - allocations: list of dicts with product, step, hours, and hours_per_workday (all numeric)
    """

    x_vars = results['x_vars']
    y_vars = results['y_vars']
    
    # Get all unique machines, products, and steps from x_vars keys
    all_machines = set()
    all_products = set()
    all_steps = set()
    
    for (m, p, s) in x_vars.keys():
        all_machines.add(m)
        all_products.add(p)
        all_steps.add(s)
    

    machine_allocation_dict = {}
    
    for machine in sorted(all_machines):
        # Extract numeric value from LpVariable object
        qty_var = y_vars[machine].varValue
        qty = int(qty_var) if qty_var else 0
        
        # Allocation for (p, s)
        allocations = []
        total_hours = 0.0
        
        for product in sorted(all_products):
            for step in sorted(all_steps):
                key = (machine, product, step)
                if key in x_vars:
                    # This to be number of hrs per week
                    hours_var = x_vars[key].varValue
                    hours = float(hours_var) if hours_var else 0.0
                    
                    if hours > 0:  # Only include non-zero allocations
                        # Calculate mins/ hr : (hours / (5*16*OEE)) * 60 / qty to per instance
                        min_per_hr = ((hours / (5*16*OEE)) * 60) / qty
                        
                        allocations.append({
                            'product': product,
                            'step': step,
                            'hours': round(hours, 4),
                            'min_per_hr_perstation': round(min_per_hr, 4)
                        })
                        total_hours += hours
        
        # Only include machines with allocations or that were purchased
        if allocations or qty > 0:
            # Calculate total hours per workday
            total_hours_per_workday = (total_hours / 5) / qty # for a single machine
            
            machine_allocation_dict[machine] = {
                'quantity': qty,
                'total_hours': round(total_hours, 4),
                'avg_total_hours_per_workday': round(total_hours_per_workday, 4), # just for snity check
                'allocations': allocations
            }
    
    return machine_allocation_dict


In [87]:
allocation_dict = get_machine_time_allocation_dict(results)

# Display a sample machine
print("Sample: Machine 'ABC' allocation:")
print(f"Quantity: {allocation_dict['ABC']['quantity']}")
print(f"Total Hours: {allocation_dict['ABC']['total_hours']}")
print(f"Total Hours per Workday: {allocation_dict['ABC']['avg_total_hours_per_workday']}")
print(f"\nFirst 5 allocations:")
for alloc in allocation_dict['ABC']['allocations'][:5]:
    print(f"  {alloc}")
    
print(f"\nTotal machines in dict: {len(allocation_dict)}")

Sample: Machine 'ABC' allocation:
Quantity: 1
Total Hours: 61.8408
Total Hours per Workday: 12.3682

First 5 allocations:
  {'product': 'P1', 'step': 'A', 'hours': 2.86, 'min_per_hr_perstation': 2.432}
  {'product': 'P1', 'step': 'C', 'hours': 23.3358, 'min_per_hr_perstation': 19.8434}
  {'product': 'P4', 'step': 'B', 'hours': 23.4783, 'min_per_hr_perstation': 19.9646}
  {'product': 'P6', 'step': 'C', 'hours': 12.1667, 'min_per_hr_perstation': 10.3458}

Total machines in dict: 17


In [ ]:
# save the result
with open("Allocation.json", "w") as f:
    json.dump(allocation_dict, f, indent=4) 